# Import modules

In [1]:
import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.Helper import (
    create_pipeline,
    load_and_split_data,
    preprocess_data,
    save_pipeline,
    predict_with_model,
    evaluate_model, 
)

# Preparation

In [2]:
# 1. Load and split data T
data_path = os.path.join("..", "data", "train.csv")
train_data, test_data = load_and_split_data(data_path)

# 2. Create pipeline
pipeline = create_pipeline()

# 3. Fit pipeline on train data only
X_train, y_train, outlier_bounds, missing_fill_strategy = preprocess_data(train_data, pipeline, train=True)

# Save the fitted pipeline (if needed)
# save_pipeline(
#     preprocessing_pipeline=pipeline,
#     outlier_bounds=outlier_bound,
#     missing_fill_strategy=missing_fill_strategy,
# )

# 4. Apply fitted pipeline and scaler to test data
X_test, y_test = preprocess_data(
    test_data,
    pipeline,
    missing_fill_strategy=missing_fill_strategy,
    outlier_bounds=outlier_bounds,
    train=False,
)

LOADING AND SPLITTING DATA


d:\Study\VPBank_Hackathon_25\src\Helper.py:100: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path)


Raw data loaded: (100000, 28)
After dropping personal info: (100000, 22)
Train data: (80000, 22)
Test data: (20000, 22)

PREPROCESSING DATA
Data after pipeline: (80000, 52)
Handling missing values with intelligent strategy...
  Monthly_Inhand_Salary: 12032 missing -> median (3191104166666666.0000) (skewed: 2.60)
  Num_of_Delayed_Payment: 5595 missing -> median (14.0000) (skewed: 14.21)
  Changed_Credit_Limit: 1691 missing -> median (902.0000) (skewed: 10.44)
  Num_Credit_Inquiries: 1549 missing -> median (60.0000) (skewed: 9.69)
  Credit_History_Age: 7240 missing -> mean (18.4345) (normal dist)
  Amount_invested_monthly: 3605 missing -> median (8131127094677352.0000) (skewed: 1.77)
  Monthly_Balance: 950 missing -> median (6580492431588418.0000) (skewed: 106.26)
Converting object columns to numeric codes...
Handling outliers...
  Annual_Income: 3409 outliers clipped
  Monthly_Inhand_Salary: 5965 outliers clipped
  Num_Bank_Accounts: 1056 outliers clipped
  Num_Credit_Card: 1776 outlier

# XGBoost

In [3]:
# Save model library
import joblib

In [4]:
from src.XGBoost import train_xgboost_model, train_xgboost_model_with_optuna

model, best_params, study = train_xgboost_model_with_optuna(X_train, y_train)

accuracy = evaluate_model(model, X_test, y_test)

joblib.dump(model, os.path.join("..", "models", "xgboost.pkl"))

c:\Users\PC\miniconda3\envs\vpbank\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-16 02:30:32,551] A new study created in memory with name: no-name-f21ac987-58cb-4051-adcf-f2109e6553fd



TRAINING XGBOOST MODEL WITH OPTUNA OPTIMIZATION
Starting hyperparameter optimization with 100 trials...


[I 2025-07-16 02:31:11,709] Trial 0 finished with value: 0.7586000113309542 and parameters: {'n_estimators': 467, 'max_depth': 11, 'learning_rate': 0.019668025817056844, 'subsample': 0.6952713431118388, 'colsample_bytree': 0.9907853317372661, 'min_child_weight': 5, 'gamma': 0.5219484736862424, 'reg_alpha': 1.0351552071372434, 'reg_lambda': 1.6938552281718375}. Best is trial 0 with value: 0.7586000113309542.
[I 2025-07-16 02:32:13,521] Trial 1 finished with value: 0.7802249885249269 and parameters: {'n_estimators': 219, 'max_depth': 17, 'learning_rate': 0.0451820456582898, 'subsample': 0.9920459633368368, 'colsample_bytree': 0.7175088057711556, 'min_child_weight': 1, 'gamma': 0.07126734815729863, 'reg_alpha': 0.9664745542294606, 'reg_lambda': 1.0088892988620595}. Best is trial 1 with value: 0.7802249885249269.
[W 2025-07-16 02:33:25,239] Trial 2 failed with parameters: {'n_estimators': 842, 'max_depth': 17, 'learning_rate': 0.02888924702337883, 'subsample': 0.5851886612671693, 'colsampl

KeyboardInterrupt: 

# Random Forest

# TabNet